# Experiment 4 — Multi-Start ICP: Effect of `n_starts`

**Goal:** Examine how the number of random starting rotations (`n_starts`) affects ICP alignment quality.

For each value of `n_starts` ∈ [1, 20] we run `MultiStartICP` over multiple random experiments and measure:
- Rotation error (°) vs. ground truth
- Translation error vs. ground truth
- Mean closest-point residual of the best trial

The shaded band shows ± 1 std across seeds.

In [1]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from icp import ICP, MultiStartICP
from experiment_runner import MultiSeedSyntheticICPResult, fit_multi_seed
from visualization import ErrorMetricsVisualizer

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Setup

We use a **clustered** point cloud (8 Gaussian blobs), which has enough geometric structure to create meaningful local minima — the scenario where multi-start ICP is most useful.

In [2]:
N_SEEDS = 10
N_STARTS_RANGE = list(range(1, 21))  # 1 … 20

# Fixed ICP configuration reused across all trials.
# record_history=False: this notebook only ever reads the final `transformation`,
# never the per-iteration cloud/matching history. Keeping those would retain a
# full point cloud + correspondence set per iteration per trial, which across a
# sweep this size (up to 20 trials x 10 seeds x 20 n_starts values) is enough
# to exhaust memory.
BASE_ICP = ICP(max_iter=200, tol=1e-10, verbose=False, record_history=False)

In [3]:
from algebra_utils import sample_dispersed_rotations, sample_uniform_rotations

EXPERIMENT_KWARGS = {
    "n": 500, "noise_std": 0.01, "t_scale": 8.0, "style": 'clustered'
}


def run_sweep(rotation_sampler=sample_dispersed_rotations) -> list[MultiSeedSyntheticICPResult]:
    """Run the n_starts sweep with a given MultiStartICP rotation_sampler.

    Args:
        rotation_sampler: Callable(n, rng) -> list of n (3, 3) SO(3) rotations,
                           passed straight through to MultiStartICP. Defaults to
                           the dispersed (farthest-point) sampler.

    Returns:
        One MultiSeedSyntheticICPResult per n_starts value in N_STARTS_RANGE.
    """
    sweep_results: list[MultiSeedSyntheticICPResult] = []
    for n_starts in tqdm(N_STARTS_RANGE, desc=f'n_starts ({rotation_sampler.__name__})'):
        multi_icp = MultiStartICP(BASE_ICP, n_starts=n_starts, seed=0, rotation_sampler=rotation_sampler)
        sweep_results.append(
            fit_multi_seed(multi_icp, list(range(N_SEEDS)), experiment_kwargs=EXPERIMENT_KWARGS, verbose=False)
        )
    return sweep_results

## 2. Data collection

For every `n_starts` value we run `fit_multi_seed`

In [4]:
results = run_sweep()

n_starts (sample_dispersed_rotations): 100%|██████████| 20/20 [01:52<00:00,  5.63s/it]


## 3. Results

Mean ± 1 std over `N_SEEDS` random experiments for each metric.

In [ ]:
rot_errors = np.array([r.rotation_errors for r in results])
t_errors = np.array([r.translation_errors for r in results])
residuals = np.array([r.mean_closest_point_residuals for r in results])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
ErrorMetricsVisualizer.plot_parameter_sweep(
    axes,
    x=N_STARTS_RANGE,
    data_per_metric=[rot_errors, t_errors, residuals],
    y_labels=['Rotation error (°)', 'Translation error', 'Mean closest-point residual'],
    x_label='n_starts',
)

plt.suptitle(
    f'Multi-Start ICP — effect of n_starts  ({N_SEEDS} seeds per value)',
    y=1.02
)
plt.tight_layout()
plt.savefig('../results/6_multi_start_icp_error_vs_n_starts.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Dispersed vs. random initial rotations

`MultiStartICP` now defaults to **greedy farthest-point (dispersed)** sampling for its starting rotations (`sample_dispersed_rotations`), instead of plain i.i.d. uniform sampling (`sample_uniform_rotations`). The hypothesis: for small `n_starts`, pure random sampling can unluckily cluster and leave gaps in SO(3), missing the basin of attraction that would have led to the correct alignment — dispersed sampling should reduce that risk.

Section 2 already ran the sweep with the (now-default) dispersed sampler, so we reuse `results` from there as the dispersed baseline and only run the sweep once more with the plain random sampler, then compare:
- Rotation / translation error and mean residual, as before, overlaid for both strategies.
- **Failure rate**: fraction of seeds per `n_starts` where the best trial's rotation error exceeds 5° — a direct "wrong basin" proxy (more interpretable than a residual threshold, since residual scale depends on `noise_std`).

In [6]:
results_dispersed = results  # already computed in section 2 with the default (dispersed) sampler
results_random = run_sweep(sample_uniform_rotations)

n_starts (sample_uniform_rotations): 100%|██████████| 20/20 [01:59<00:00,  5.98s/it]


In [ ]:
FAILURE_ROTATION_THRESHOLD_DEG = 5.0


def failure_rate(sweep_results: list[MultiSeedSyntheticICPResult]) -> np.ndarray:
    """Fraction of seeds per n_starts whose best trial missed the correct basin.

    Args:
        sweep_results: One MultiSeedSyntheticICPResult per n_starts value.

    Returns:
        Array (len(sweep_results),) of failure fractions in [0, 1].
    """
    return np.array([
        np.mean(np.array(r.rotation_errors) > FAILURE_ROTATION_THRESHOLD_DEG)
        for r in sweep_results
    ])


rot_errors_dispersed = np.array([r.rotation_errors for r in results_dispersed])
t_errors_dispersed = np.array([r.translation_errors for r in results_dispersed])
residuals_dispersed = np.array([r.mean_closest_point_residuals for r in results_dispersed])
failure_dispersed = failure_rate(results_dispersed)

rot_errors_random = np.array([r.rotation_errors for r in results_random])
t_errors_random = np.array([r.translation_errors for r in results_random])
residuals_random = np.array([r.mean_closest_point_residuals for r in results_random])
failure_random = failure_rate(results_random)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

METRIC_LABELS = ['Rotation error (°)', 'Translation error', 'Mean closest-point residual']
ErrorMetricsVisualizer.plot_parameter_sweep(
    axes[:3], x=N_STARTS_RANGE,
    data_per_metric=[rot_errors_dispersed, t_errors_dispersed, residuals_dispersed],
    y_labels=METRIC_LABELS, x_label='n_starts',
    colors=['tab:blue'] * 3, label='dispersed',
)
ErrorMetricsVisualizer.plot_parameter_sweep(
    axes[:3], x=N_STARTS_RANGE,
    data_per_metric=[rot_errors_random, t_errors_random, residuals_random],
    y_labels=METRIC_LABELS, x_label='n_starts',
    colors=['tab:orange'] * 3, label='random',
)

axes[3].plot(N_STARTS_RANGE, failure_dispersed, marker='o', ms=5, color='tab:blue', linewidth=2, label='dispersed')
axes[3].plot(N_STARTS_RANGE, failure_random, marker='o', ms=5, color='tab:orange', linewidth=2, label='random')
axes[3].set_xlabel('n_starts')
axes[3].set_ylabel(f'Failure rate (rot. error > {FAILURE_ROTATION_THRESHOLD_DEG:.0f}°)')
axes[3].set_title('Failure rate')
axes[3].set_xticks(N_STARTS_RANGE)
axes[3].legend()

plt.suptitle(
    f'Multi-Start ICP — dispersed vs. random initial rotations  ({N_SEEDS} seeds per value)',
    y=1.02
)
plt.tight_layout()
plt.savefig('../results/6_multi_start_icp_dispersed_vs_random.png', dpi=150, bbox_inches='tight')
plt.show()

### Observations

Dispersed sampling's advantage is concentrated in a mid-range "sweet spot" (`n_starts` ≈ 6–11), not spread evenly across the sweep:

- **`n_starts = 1`**: both strategies are identical by construction (a single start is a single start, whichever sampler picks it) — both fail on essentially every seed (failure rate ≈ 1.0, rotation error ≈ 145°).
- **`n_starts ≈ 6–11`**: this is where dispersion pays off most. Failure rate drops to ~0.1–0.2 for dispersed vs. a fairly flat ~0.4 for random — roughly half the failures. Mean point residual and rotation/translation error follow the same pattern (e.g. at `n_starts=7`: residual ≈ 0.3 dispersed vs. ≈ 1.25 random). This is exactly the regime we hypothesized: too few starts for random sampling to reliably stumble into the right basin, but enough that spreading them out deterministically covers it.
- **`n_starts ≥ 18`**: both strategies converge to ≈0 failure rate and near-zero error — with that many starts, even random sampling covers SO(3) densely enough that the basin is essentially always hit.
- Some non-monotonicity in both curves (e.g. the failure-rate spike at `n_starts=12`) is sampling noise — only `N_SEEDS=10` seeds per point, so each step is worth 10 percentage points; a larger `N_SEEDS` would smooth this out.

**Takeaway:** dispersed sampling doesn't help everywhere — it specifically buys you a lower `n_starts` budget for the same reliability. If cost/runtime matters, this sweep suggests `n_starts ≈ 7–10` with dispersed sampling reaches roughly the same failure rate that random sampling only reaches around `n_starts ≈ 15–18`, i.e. it lets you use fewer, cheaper ICP trials for the same quality.